# Partitioning

A stream has three groups of axes, `(S*, T, D*)`: sequence steps, the samples
of its window, its features (see [Model design](01_model_design.ipynb)). The
partitioning layers take parts of a stream along one group, or join streams
along it - there is no indexing `x[0]` on streams:

| Group | Pick one | Take a range | Join | Reverse |
|---|---|---|---|---|
| dims `D*` | `Select` | `Range` | `Concatenate` | - |
| window `T` | `TimeSelect` | `TimeRange` | `TimeConcatenate` | `TimeReverse` |
| sequence `S*` | `SeqSelect` | `SeqRange` | `SeqConcatenate`, `SeqStack` | `SeqReverse` |

This tutorial applies each one to signals whose values are counters, so that
what is taken can be read off the numbers, and ends with the patterns they are
used for: targets one step ahead, differences, feature groups, trajectories.

In [1]:
import os

# Pick the Keras backend before nnodely (or Keras) is imported:
# "tensorflow" (the default), "torch" or "jax".
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
import numpy as np

from nnodely import (
    Concatenate,
    Input,
    Max,
    Mean,
    Modely,
    Output,
    Range,
    Select,
    SeqConcatenate,
    SeqRange,
    SeqReverse,
    SeqSelect,
    SeqStack,
    Sum,
    TimeConcatenate,
    TimeRange,
    TimeReverse,
    TimeSelect,
)


def evaluate(data, **streams):
    """Run the given streams on one sample of data, and print each one: its
    name, its shape, and its values, with the axes of length 1 left out."""
    probe = Modely(
        "probe", inputs=list(data), outputs=[Output(k, s) for k, s in streams.items()]
    ).build()
    results = probe({node.name: value[None] for node, value in data.items()})
    for name, value in results.items():
        values = (
            np.squeeze(keras.ops.convert_to_numpy(value)[0])
            .astype(float)
            .round(4)
            .tolist()
        )
        print(f"{name:12s} {str(streams[name].shape):28s} {values}")

## The dims: `Select`, `Range`, `Concatenate`

`v` has a window of 3 samples of 4 features. Its values count `0..11`, a row per
sample of the window, oldest first:

In [2]:
v = Input("v", dim=4, tw=3)
v_data = np.arange(12.0).reshape(3, 4)  # (T, D) for one sample
print(v_data)

[[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]]


- `Select(i)` picks one feature: the dim axis is dropped (a vector becomes one
  feature, `D=(1,)`);
- `Range(start, end)` keeps the features `start..end-1`;
- `Concatenate()` joins streams along the dims; the windows must match.

The window is untouched: each sample is partitioned the same way. Each line
`evaluate` prints is one stream: its name, its shape, and its values for this
sample.

In [3]:
evaluate(
    {v: v_data},
    second=Select(1)(v),
    last=Select(-1)(v),
    middle=Range(1, 3)(v),
    joined=Concatenate()([Select(3)(v), Range(0, 2)(v)]),
)

second       Shape(S=(), T=3, D=(1,))     [1.0, 5.0, 9.0]
last         Shape(S=(), T=3, D=(1,))     [3.0, 7.0, 11.0]
middle       Shape(S=(), T=3, D=(2,))     [[1.0, 2.0], [5.0, 6.0], [9.0, 10.0]]
joined       Shape(S=(), T=3, D=(3,))     [[3.0, 0.0, 1.0], [7.0, 4.0, 5.0], [11.0, 8.0, 9.0]]


A matrix-valued signal has several dim axes; `axis` picks the one to act on.
`Select` drops that axis, unless `keep_dim=True` keeps it with length 1:

In [4]:
m = Input("m", dim=(2, 3), tw=1)
m_data = np.arange(6.0).reshape(1, 2, 3)
evaluate(
    {m: m_data},
    row_1=Select(1)(m),  # axis 0: the second row
    column_2=Select(2, axis=1)(m),  # axis 1: the third column
    row_1_kept=Select(1, keep_dim=True)(m),
    columns_0_1=Range(0, 2, axis=1)(m),
)

row_1        Shape(S=(), T=1, D=(3,))     [3.0, 4.0, 5.0]
column_2     Shape(S=(), T=1, D=(2,))     [2.0, 5.0]
row_1_kept   Shape(S=(), T=1, D=(1, 3))   [3.0, 4.0, 5.0]
columns_0_1  Shape(S=(), T=1, D=(2, 2))   [[0.0, 1.0], [3.0, 4.0]]


The reductions `Sum`, `Mean`, `Max`, `Min` and `Norm` collapse the dims (or the
one `axis` given) into one value per sample, keeping the axis with length 1:

In [5]:
evaluate({v: v_data}, total=Sum()(v), average=Mean()(v), largest=Max()(v))

total        Shape(S=(), T=3, D=(1,))     [6.0, 22.0, 38.0]
average      Shape(S=(), T=3, D=(1,))     [1.5, 5.5, 9.5]
largest      Shape(S=(), T=3, D=(1,))     [3.0, 7.0, 11.0]


## The window: `TimeSelect`, `TimeRange`, `TimeConcatenate`, `TimeReverse`

`w` is a window of 5 samples, `[0, 1, 2, 3, 4]`, oldest first: index `-1` is the
newest sample.

- `TimeSelect(i)` keeps one sample, as a window of 1, `T=1`;
- `TimeSelect(i, keep_tw=False)` keeps it **without** a window, so that it
  applies to every sample of a window it is combined with;
- `TimeRange(start, end)` keeps samples `start..end-1` (negative bounds count
  from the end, `None` leaves a bound open);
- `TimeConcatenate()` joins windows; `TimeReverse()` puts the newest first.

In [6]:
w = Input("w", tw=5)
w_data = np.arange(5.0).reshape(5, 1)
evaluate(
    {w: w_data},
    newest=TimeSelect(-1)(w),
    oldest_two=TimeRange(0, 2)(w),
    last_three=TimeRange(-3, None)(w),
    joined=TimeConcatenate()([TimeRange(3, 5)(w), TimeRange(0, 2)(w)]),
    reversed=TimeReverse()(w),
)

newest       Shape(S=(), T=1, D=(1,))     4.0
oldest_two   Shape(S=(), T=2, D=(1,))     [0.0, 1.0]
last_three   Shape(S=(), T=3, D=(1,))     [2.0, 3.0, 4.0]
joined       Shape(S=(), T=4, D=(1,))     [3.0, 4.0, 0.0, 1.0]
reversed     Shape(S=(), T=5, D=(1,))     [4.0, 3.0, 2.0, 1.0, 0.0]


`T=1` is a window like any other, and does not combine with a longer one: to
subtract the newest sample from each sample of the window, drop its window:

In [7]:
try:
    w - TimeSelect(-1)(w)
except ValueError as error:
    print("ValueError:", str(error).split(":")[1].strip())

evaluate({w: w_data}, relative=w - TimeSelect(-1, keep_tw=False)(w))

ValueError: streams with windows of 1 and 5 cannot be combined
relative     Shape(S=(), T=5, D=(1,))     [-4.0, -3.0, -2.0, -1.0, 0.0]


## The sequence: `SeqSelect`, `SeqRange`, `SeqStack`, `SeqConcatenate`, `SeqReverse`

A sequence holds several steps, each a window: the trajectories of recurrent
models (see [Recurrent models](10_recurrent_loop.ipynb)). `s` is 4 steps of one
sample, counting `0..3`:

- `SeqSelect(i)` keeps one step: the sequence axis is **removed**;
- `SeqRange(start, end)` keeps a range of steps (one step long, the axis is
  removed too);
- `SeqConcatenate()` joins along an existing sequence axis;
- `SeqStack()` stacks streams of equal shape into a **new** sequence axis;
- `SeqReverse()` reverses the steps.

In [8]:
s = Input("s", seq=4, tw=1)
s_data = np.arange(4.0).reshape(4, 1, 1)  # (S, T, D)
evaluate(
    {s: s_data},
    last_step=SeqSelect(-1)(s),
    middle=SeqRange(1, 3)(s),
    doubled=SeqConcatenate()([s, s]),
    stacked=SeqStack()([SeqSelect(0)(s), SeqSelect(3)(s)]),
    reversed=SeqReverse()(s),
)

last_step    Shape(S=(), T=1, D=(1,))     3.0
middle       Shape(S=(2,), T=1, D=(1,))   [1.0, 2.0]
doubled      Shape(S=(8,), T=1, D=(1,))   [0.0, 1.0, 2.0, 3.0, 0.0, 1.0, 2.0, 3.0]
stacked      Shape(S=(2,), T=1, D=(1,))   [0.0, 3.0]
reversed     Shape(S=(4,), T=1, D=(1,))   [3.0, 2.0, 1.0, 0.0]


With several sequence axes, `axis` counts them outermost first: `axis=0` is the
outermost - the rollout of the last `Loop` applied.

In [9]:
grid = Input("grid", seq=(2, 3), tw=1)
grid_data = np.arange(6.0).reshape(2, 3, 1, 1)
evaluate(
    {grid: grid_data},
    outer_1=SeqSelect(1)(grid),  # the second of the 2 outer steps
    inner_0=SeqSelect(0, axis=1)(grid),  # the first inner step of each
)

outer_1      Shape(S=(3,), T=1, D=(1,))   [3.0, 4.0, 5.0]
inner_0      Shape(S=(2,), T=1, D=(1,))   [0.0, 3.0]


## Patterns

**A target one step ahead.** The input reads the past and the next sample:
`tw=(n, 1)`. The model reads the past, `TimeRange(0, n)`, and the next sample
is the target, `TimeSelect(-1)`:

```python
y = Input("y", tw=(10, 1))
prediction = Output("y_next", Fir(features=1)([TimeRange(0, 10)(y)]))
model.minimize("one_step", prediction, TimeSelect(-1)(y))
```

**A difference.** A velocity from two positions, newest first:

In [10]:
x = Input("x", tw=2)
velocity = (TimeSelect(-1)(x) - TimeSelect(-2)(x)) / 0.1
evaluate({x: np.array([[1.0], [1.5]])}, velocity=velocity)

velocity     Shape(S=(), T=1, D=(1,))     5.0


**Feature groups.** A state vector `[position, velocity]` split into its parts,
processed, and put back together:

In [11]:
state = Input("state", dim=2, tw=1)
position, speed = Select(0)(state), Select(1)(state)
next_state = Concatenate()([position + 0.1 * speed, 0.9 * speed])
evaluate({state: np.array([[2.0, 1.0]])}, next_state=next_state)

next_state   Shape(S=(), T=1, D=(2,))     [2.1, 0.9]


## Summary

| Take | Code | Shape |
|---|---|---|
| a feature | `Select(i)(x)`, `Select(i, axis=k, keep_dim=True)` | `D=(4,)` -> `D=(1,)` |
| features | `Range(a, b)(x)` | `D=(4,)` -> `D=(b-a,)` |
| join features | `Concatenate()([a, b])` | dims added up |
| reduce features | `Sum()`, `Mean()`, `Max()`, `Min()`, `Norm()` | -> `D=(1,)` |
| a sample | `TimeSelect(i)(x)`; `keep_tw=False` drops the window | `T=5` -> `T=1` / `None` |
| samples | `TimeRange(a, b)(x)` | `T=5` -> `T=b-a` |
| join / flip windows | `TimeConcatenate()`, `TimeReverse()` | |
| a step | `SeqSelect(i)(s)` | `S=(4,)` -> `S=()` |
| steps | `SeqRange(a, b)(s)` | `S=(4,)` -> `S=(b-a,)` |
| join / stack / flip steps | `SeqConcatenate()`, `SeqStack()`, `SeqReverse()` | |

Next: [Fir and Linear](08_fir_linear.ipynb).